# Lesson 2: Data Wrangling — JSON to DataFrame

**Week 4 · Data Engineering Course**

---

In Lesson 1 you learned how to call an API and get a JSON response. That response is nested — it has dicts inside dicts, and lists of values rather than a ready-made table.

This lesson teaches you how to:
- Flatten a nested JSON response into a pandas DataFrame
- Combine results from multiple cities into one table
- Validate the data (check for missing values, wrong types)
- Handle pagination for APIs that return data in pages
- Save the final table to CSV

In [ ]:
import json
import time
import requests
import pandas as pd
from pathlib import Path

DATA  = Path('data')
CLEAN = DATA / 'clean'
CLEAN.mkdir(parents=True, exist_ok=True)

print('Ready.')

---

## 2.1 Understanding the JSON Structure

Before writing any code, look at the response you are working with.

In [ ]:
# Load the sample response saved in Lesson 1
# (or load the one in data/ if you have not done Lesson 1 yet)
try:
    with open(DATA / 'lagos_forecast.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)
except FileNotFoundError:
    with open(DATA / 'sample_weather.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

print(json.dumps(raw, indent=2)[:800], '...')   # first 800 characters

In [ ]:
# The data we want is in raw['daily']
# It is a dict of lists — each key is a column, each list is the values

daily = raw['daily']
print('Keys in daily:', list(daily.keys()))
print('Number of rows:', len(daily['time']))

# Each list has the same length — one value per day
for key, values in daily.items():
    print(f'  {key}: {values[:3]} ...')

---

## 2.2 Flattening JSON into a DataFrame

`pd.DataFrame()` can build a table directly from a dict of lists — exactly the format Open-Meteo returns.

In [ ]:
# Build a DataFrame from the daily dict
df = pd.DataFrame(raw['daily'])

print(df.shape)
df.head()

In [ ]:
# Rename columns to cleaner names
df = df.rename(columns={
    'time':                'date',
    'temperature_2m_max':  'temp_max_c',
    'temperature_2m_min':  'temp_min_c',
    'precipitation_sum':   'rain_mm',
    'windspeed_10m_max':   'wind_max_kmh',
})

# Convert the date column from string to proper datetime
df['date'] = pd.to_datetime(df['date'])

print(df.dtypes)
df.head()

In [ ]:
# Add metadata from the top-level response
# This tells you which city the rows belong to
df['latitude']  = raw['latitude']
df['longitude'] = raw['longitude']
df['timezone']  = raw['timezone']

df.head()

---

## 2.3 Multiple Cities — Fetch, Flatten, Combine

A real pipeline does not just fetch one city. It loops over all of them and concatenates the results.

In [ ]:
# Reusable fetch function from Lesson 1
def fetch_weather(city_name, latitude, longitude, timezone):
    '''Fetch 7-day forecast. Returns a DataFrame, or None on failure.'''
    params = {
        'latitude':  latitude,
        'longitude': longitude,
        'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max',
        'timezone':  timezone,
    }
    try:
        response = requests.get(
            'https://api.open-meteo.com/v1/forecast',
            params=params,
            timeout=10,
        )
        response.raise_for_status()
        raw = response.json()
    except requests.RequestException as e:
        print(f'  SKIP {city_name}: {e}')
        return None

    df = pd.DataFrame(raw['daily']).rename(columns={
        'time':                'date',
        'temperature_2m_max':  'temp_max_c',
        'temperature_2m_min':  'temp_min_c',
        'precipitation_sum':   'rain_mm',
        'windspeed_10m_max':   'wind_max_kmh',
    })
    df['date']      = pd.to_datetime(df['date'])
    df['city']      = city_name
    df['latitude']  = latitude
    df['longitude'] = longitude
    df['timezone']  = timezone
    return df

print('fetch_weather() defined.')

In [ ]:
# Load cities from the CSV
cities_df = pd.read_csv(DATA / 'cities.csv')
print(cities_df)

all_frames = []

for _, row in cities_df.iterrows():
    print(f'Fetching {row["city"]}...')
    df = fetch_weather(row['city'], row['latitude'], row['longitude'], row['timezone'])
    if df is not None:
        all_frames.append(df)
    time.sleep(0.3)   # be polite — do not hammer the server

if all_frames:
    weather = pd.concat(all_frames, ignore_index=True)
    print(f'\nCombined: {weather.shape}')
else:
    print('No data fetched.')

In [ ]:
# Check the combined table
print(weather['city'].value_counts())
print()
weather.head(10)

---

## 2.4 Validating API Data

API data can have missing values, wrong types, or values that are out of range. Check before you save.

In [ ]:
# Missing values
print('Missing values per column:')
print(weather.isnull().sum())

In [ ]:
# Check column types
print(weather.dtypes)

In [ ]:
# Sanity checks on numeric ranges
# Temperatures in Africa should be roughly -5°C to 50°C
# Precipitation should be >= 0
# Wind should be >= 0

def validate_weather(df):
    issues = []

    if df['temp_max_c'].lt(-10).any() or df['temp_max_c'].gt(60).any():
        issues.append('temp_max_c outside expected range (-10 to 60)')

    if df['temp_min_c'].lt(-10).any() or df['temp_min_c'].gt(60).any():
        issues.append('temp_min_c outside expected range (-10 to 60)')

    if df['rain_mm'].lt(0).any():
        issues.append('rain_mm has negative values')

    if df['wind_max_kmh'].lt(0).any():
        issues.append('wind_max_kmh has negative values')

    missing = df.isnull().sum()
    for col, count in missing.items():
        if count > 0:
            issues.append(f'{col} has {count} missing values')

    if issues:
        print('Validation issues found:')
        for issue in issues:
            print(f'  - {issue}')
    else:
        print('Validation passed — no issues found.')

validate_weather(weather)

In [ ]:
# Fill any missing precipitation with 0 (NaN usually means no rain was recorded)
weather['rain_mm'] = weather['rain_mm'].fillna(0.0)

# Round numeric columns to 2 decimal places
numeric_cols = ['temp_max_c', 'temp_min_c', 'rain_mm', 'wind_max_kmh']
weather[numeric_cols] = weather[numeric_cols].round(2)

print('After fill and round:')
weather.describe()

---

## 2.5 Pagination

Many APIs cannot return all their data in one request — the result could be millions of rows. Instead they return one **page** at a time and tell you how to get the next one.

The two most common pagination patterns:

### Pattern 1 — Page number

```python
# API returns: {'data': [...], 'total_pages': 5, 'page': 1}
all_rows = []
page = 1

while True:
    response = requests.get(url, params={..., 'page': page})
    data = response.json()
    all_rows.extend(data['data'])
    if page >= data['total_pages']:   # no more pages
        break
    page += 1
    time.sleep(0.2)   # rate limiting
```

### Pattern 2 — next_url cursor

```python
# API returns: {'data': [...], 'next': 'https://api.example.com/data?cursor=abc'}
all_rows = []
next_url = 'https://api.example.com/data'

while next_url:
    response = requests.get(next_url)
    data = response.json()
    all_rows.extend(data['data'])
    next_url = data.get('next')   # None when there are no more pages
    time.sleep(0.2)
```

Open-Meteo does not paginate — you fetch a date range in one request — but you will encounter both patterns in real projects.

In [ ]:
# Simulate a paginated API with a simple mock
def mock_paginated_api(page):
    '''Pretend this is an API that returns one page of data at a time.'''
    pages = {
        1: {'data': [{'id': 1, 'value': 'a'}, {'id': 2, 'value': 'b'}], 'total_pages': 3, 'page': 1},
        2: {'data': [{'id': 3, 'value': 'c'}, {'id': 4, 'value': 'd'}], 'total_pages': 3, 'page': 2},
        3: {'data': [{'id': 5, 'value': 'e'}],                          'total_pages': 3, 'page': 3},
    }
    return pages.get(page)

# Fetch all pages
all_rows = []
current_page = 1

while True:
    data = mock_paginated_api(current_page)
    if data is None:
        break
    all_rows.extend(data['data'])
    print(f'Page {current_page}: got {len(data["data"])} rows')
    if current_page >= data['total_pages']:
        break
    current_page += 1

print(f'\nTotal rows collected: {len(all_rows)}')
print(all_rows)

---

## 2.6 Saving to CSV

In [ ]:
# Reorder columns for readability
cols = ['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm', 'wind_max_kmh',
        'latitude', 'longitude', 'timezone']
weather = weather[cols]

out_path = CLEAN / 'weather_forecast.csv'
weather.to_csv(out_path, index=False)
print(f'Saved {len(weather)} rows to {out_path}')

# Verify
check = pd.read_csv(out_path)
print(f'Re-read: {check.shape}')
check.head()

In [ ]:
# Quick analysis on the combined data
print('Hottest city (max temperature this week):')
print(weather.groupby('city')['temp_max_c'].max().sort_values(ascending=False))

print('\nWettest city (total rain this week):')
print(weather.groupby('city')['rain_mm'].sum().sort_values(ascending=False).round(1))

---

## Key Takeaways

1. `pd.DataFrame(dict_of_lists)` builds a table directly from the format most APIs return — a dict where each key is a column and each value is a list.
2. Add metadata columns (`city`, `latitude`, `timezone`) to the DataFrame before concatenating so every row knows where it came from.
3. `pd.concat(list_of_frames, ignore_index=True)` stacks multiple DataFrames into one table.
4. Always **validate** API data: check for missing values, unexpected nulls, and values outside realistic ranges.
5. **Pagination**: most APIs return data in pages. Loop with a `while True` until there are no more pages. Add a small `time.sleep()` between requests to avoid hitting rate limits.
6. Save the combined, validated table to CSV with `index=False` as the checkpoint before loading into a database.